# Smart MCQ Solver

**Objective**: The goal of this competition is to build a machine learning pipeline capable of answering complex multiple-choice questions. 

Given a question prompt and five possible options (A, B, C, D, E), the model must rank the top 3 most likely answers. The evaluation metric for the competition is **Mean Average Precision at 3 (MAP@3)**, meaning models are rewarded heavily for getting the correct answer as their #1 prediction.

---

## Section 1: Project Setup & Data Ingestion
*Initializing the environment and loading the competition datasets.*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load datasets (Kaggle Environment Paths)
train_path = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
test_path = '/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv'

# Explicitly cast text columns to string to prevent NaN parsing issues
dtype_dict = {'prompt': str, 'A': str, 'B': str, 'C': str, 'D': str, 'E': str, 'answer': str}
train_df = pd.read_csv(train_path, dtype=dtype_dict)
test_df = pd.read_csv(test_path, dtype=dtype_dict)

# Force ID column to string just in case
if 'id' in train_df.columns:
    train_df['id'] = train_df['id'].astype(str)
if 'id' in test_df.columns:
    test_df['id'] = test_df['id'].astype(str)

print(f"Training Set Shape: {train_df.shape}")
print(f"Test Set Shape: {test_df.shape}")
print("\nData Types:")
print(train_df.dtypes)

## Section 2: Global Data Cleaning
*Checking for missing values and duplicates across the entire dataset before applying any model-specific logic.*

In [ ]:
# 1. Missing value analysis
print("Missing Values:\n", train_df.isnull().sum())


In [ ]:
# 2. Duplicate Check
train_dups = train_df.duplicated(subset=['prompt']).sum()
test_dups = test_df.duplicated(subset=['prompt']).sum()
print(f"Found {train_dups} duplicate prompts in training set.")
print(f"Found {test_dups} duplicate prompts in test set.")

### Conclusion: 
We must keep these duplicates. In reading comprehension datasets, it is common to have the exact same reading passage (prompt) asked with different options or questions appended to it. Removing them would destroy valuable training data.

## Section 3: Exploratory Data Analysis (EDA)
*Visualizing class distributions, text lengths, and option variances.*

In [ ]:
sns.set_theme(style="whitegrid")

# Plot 1: Answer Distribution
plt.figure(figsize=(8, 4))
sns.countplot(data=train_df, x='answer', order=['A', 'B', 'C', 'D', 'E'], palette='viridis')
plt.title('Target Class Distribution')
plt.show()

In [ ]:
# Plot 2: Prompt Word Count Distribution (KDE)
train_df['prompt_word_count'] = train_df['prompt'].apply(lambda x: len(str(x).split()))
plt.figure(figsize=(8, 4))
sns.histplot(data=train_df, x='prompt_word_count', bins=30, kde=True, color='blue')
plt.title('Prompt Length (Word Count) Distribution')
plt.show()

In [ ]:
# Plot 3: Option Lengths (A vs B vs C vs D vs E)
option_lengths = []
for opt in ['A', 'B', 'C', 'D', 'E']:
    lengths = train_df[opt].apply(lambda x: len(str(x).split()))
    for l in lengths:
        option_lengths.append({'Option': opt, 'Word Count': l})
opt_df = pd.DataFrame(option_lengths)

plt.figure(figsize=(8, 4))
sns.boxplot(data=opt_df, x='Option', y='Word Count', palette='pastel')
plt.title('Word Counts Across Options')
plt.ylim(0, 50) # Capped for readability
plt.show()

In [ ]:
# Plot 4: Does prompt length affect the correct answer?
plt.figure(figsize=(8, 4))
sns.boxplot(data=train_df, x='answer', y='prompt_word_count', order=['A', 'B', 'C', 'D', 'E'], palette='muted')
plt.title('Prompt Length vs. Correct Answer')
plt.show()

## Section 4: Shared Utilities
*Foundational helper functions for evaluation, data formatting, and validation splits.*

### Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

# Create reproducible validation split
train_split, val_split = train_test_split(train_df, test_size=0.2, random_state=42, stratify=train_df['answer'])
print(f"Train Split: {len(train_split)} rows")
print(f"Validation Split: {len(val_split)} rows")

### Text Formatting Helpers
*Extracting options and building concatenated strings cleanly so we don't have messy inline loops later.*

In [ ]:
def get_options_list(row):
    """Extracts the 5 options from a dataframe row into a clean list."""
    return [str(row[opt]) for opt in ['A', 'B', 'C', 'D', 'E']]

def format_multiple_choice(prompt, option):
    """Concatenates a prompt and an option safely."""
    return f"{str(prompt).strip()} {str(option).strip()}"

### Evaluation Metric (MAP@3)

In [ ]:
def calculate_map3(predictions, ground_truth):
    """Calculates Mean Average Precision @ 3."""
    scores = []
    for pred_str, gt in zip(predictions, ground_truth):
        pred_list = pred_str.strip().split()
        score = 0.0
        num_correct = 0
        for k, label in enumerate(pred_list[:3], start=1):
            if label == gt:
                num_correct += 1
                score += num_correct / k
        scores.append(score)
    return np.mean(scores)

# Quick sanity check to ensure the metric works as expected
dummy_preds = ['A B C', 'D A E', 'C B A']
dummy_truths = ['A', 'A', 'C']
print(f"Sanity Check MAP@3 Score: {calculate_map3(dummy_preds, dummy_truths)}")

## Section 5: Model 1 (TF-IDF Baseline)
*Unlike Transformer models which rely on case and punctuation for semantic context, TF-IDF is a lexical Bag-of-Words model. We must aggressively normalize the text specifically for this model, and then compute cosine similarity.*

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# A standard list of English stopwords that confuse lexical TF-IDF matching
STOPWORDS = set([
    'the', 'a', 'an', 'and', 'or', 'but', 'if', 'because', 'as', 'what', 
    'when', 'where', 'how', 'which', 'who', 'whom', 'this', 'that', 'these', 
    'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 
    'has', 'had', 'do', 'does', 'did', 'to', 'from', 'in', 'out', 'on', 'off', 
    'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'all', 
    'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 
    'nor', 'not', 'only', 'own', 'same', 'so', 'than', 'too', 'very', 's', 't', 
    'can', 'will', 'just', 'don', 'should', 'now', 'of', 'for', 'with', 'about', 
    'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 
    'below', 'up', 'down', 'by'
])

def clean_text_for_tfidf(text):
    """Aggressive cleaning strictly for TF-IDF (lowercasing, removing punctuation, stopwords)."""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    
    words = text.split()
    words = [w for w in words if w not in STOPWORDS]
    return " ".join(words).strip()

print("Original:", train_df['prompt'].iloc[0][:100])
print("Cleaned TF-IDF:", clean_text_for_tfidf(train_df['prompt'].iloc[0][:100]))

In [ ]:
# --- Naive TF-IDF Baseline Evaluation --- #
tfidf_predictions = []

for idx, row in val_split.iterrows():
    # Clean the prompt
    prompt_clean = clean_text_for_tfidf(row['prompt'])
    
    # Extract and clean options using our new helper
    raw_opts = get_options_list(row)
    opts_clean = [clean_text_for_tfidf(opt) for opt in raw_opts]
    
    # Fit a new vectorizer on just this question's vocabulary
    vec = TfidfVectorizer()
    try:
        tfidf_matrix = vec.fit_transform([prompt_clean] + opts_clean)
        sims = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()
        
        top_indices = np.argsort(sims)[::-1][:3]
        pred_str = " ".join([['A', 'B', 'C', 'D', 'E'][i] for i in top_indices])
    except ValueError:
        # Fallback if text is completely empty after cleaning
        pred_str = "A B C"
        
    tfidf_predictions.append(pred_str)

val_truths = val_split['answer'].tolist()
tfidf_map3 = calculate_map3(tfidf_predictions, val_truths)

print(f"\n>>> TF-IDF MAP@3 Score: {tfidf_map3:.4f} <<<")

### Conclusion & Failed Experiment:
Our initial naive TF-IDF baseline achieved a MAP@3 of **`0.3117`**. 
We attempted to improve this by removing common English stopwords, assuming they were adding noise to the cosine similarity calculations. However, the score aggressively dropped to **`~0.27`**!

**Why?** In the context of reading comprehension and MCQs, seemingly "noisy" words like *"not"*, *"is"*, or *"because"* actually carry critical semantic signals required to match the prompt to the exact option. Removing them destroyed the context. We have kept the failed code above to document the attempt, but this proves that lexical matching (TF-IDF) is far too brittle for this task. We need semantic embeddings.